In [ ]:
import logging
import os
import random
import hashlib
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

# ── Logging setup ──────────────────────────────────────────────────────────
logging.basicConfig(
    filename="data_creation.log",
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)
log.info("=== credit_card_creation.py started ===")

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
log.info(f"Random seed set to {SEED}")

# ── Configuration ──────────────────────────────────────────────────────────
N_CUSTOMERS    = 1_000
N_MERCHANTS    = 800
N_TRANSACTIONS = 10_000_000    # 10M rows → >1 GB as Parquet
FRAUD_RATE     = 0.02          # ~2 % – consistent with Fed Reserve (2022)
OUTPUT_DIR     = "./data"
BATCH_SIZE     = 500_000       # write transactions in batches to avoid OOM

# ── Reference pools ────────────────────────────────────────────────────────
FIRST_M = ["James","John","Robert","Michael","William","David","Richard","Joseph",
           "Thomas","Charles","Christopher","Daniel","Matthew","Anthony","Mark",
           "Donald","Steven","Paul","Andrew","Joshua","Kenneth","Kevin","Brian","George"]
FIRST_F = ["Mary","Patricia","Jennifer","Linda","Barbara","Susan","Dorothy","Karen",
           "Nancy","Lisa","Betty","Margaret","Sandra","Ashley","Kimberly","Emily",
           "Donna","Michelle","Carol","Amanda","Melissa","Deborah","Stephanie","Sarah"]
LAST    = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis",
           "Rodriguez","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson",
           "Thomas","Taylor","Moore","Jackson","Martin","Lee","Perez","Thompson","White"]
STREETS = ["Main St","Oak Ave","Maple Dr","Cedar Ln","Pine Rd","Elm St",
           "Washington Blvd","Park Ave","Lake Dr","River Rd","Hill St","Forest Ave"]
CITIES  = [("New York","NY"),("Los Angeles","CA"),("Chicago","IL"),("Houston","TX"),
           ("Phoenix","AZ"),("Philadelphia","PA"),("San Antonio","TX"),("San Diego","CA"),
           ("Dallas","TX"),("San Jose","CA"),("Austin","TX"),("Jacksonville","FL"),
           ("Fort Worth","TX"),("Columbus","OH"),("Charlotte","NC"),("Indianapolis","IN"),
           ("Seattle","WA"),("Denver","CO"),("Nashville","TN"),("Oklahoma City","OK")]
JOBS    = ["Engineer","Teacher","Nurse","Manager","Analyst","Developer","Accountant",
           "Doctor","Lawyer","Consultant","Designer","Scientist","Architect","Pharmacist",
           "Technician","Administrator","Director","Coordinator","Specialist","Professor"]
CATEGORIES = ["grocery_pos","gas_transport","home","shopping_net","entertainment",
              "food_dining","personal_care","health_fitness","kids_pets","shopping_pos",
              "misc_net","misc_pos","travel","education"]
MERCH_PFX  = ["Acme","Star","Blue","Green","Metro","City","Prime","Eagle","Pacific",
              "National","Summit","Apex","Atlas","Titan","Pioneer","Silver","Golden"]
MERCH_SFX  = ["Mart","Store","Shop","Center","Hub","Plus","Express","Direct","Pro","Co"]
CARD_TYPES  = ["Visa","Mastercard","Amex","Discover"]
CARD_WGTS   = [0.45, 0.35, 0.12, 0.08]


# ── Helper ─────────────────────────────────────────────────────────────────
def make_output_dir(path: str) -> None:
    """Create output directory if it doesn't exist."""
    try:
        os.makedirs(path, exist_ok=True)
        log.info(f"Output directory ready: {path}")
    except OSError as e:
        log.error(f"Could not create output directory: {e}")
        raise


# ── Table generators ───────────────────────────────────────────────────────
def generate_customers(n: int) -> pd.DataFrame:
    """
    Generate n synthetic cardholders.
    Fields: customer_id, first, last, gender, dob, street, city, state,
            zip, job, city_pop, lat, long
    """
    log.info(f"Generating {n} customers …")
    rows = []
    for i in range(1, n + 1):
        gender = random.choice(["M", "F"])
        first  = random.choice(FIRST_M if gender == "M" else FIRST_F)
        dob    = datetime(random.randint(1950, 2000),
                          random.randint(1, 12),
                          random.randint(1, 28))
        city, state = random.choice(CITIES)
        rows.append({
            "customer_id": i,
            "first":       first,
            "last":        random.choice(LAST),
            "gender":      gender,
            "dob":         dob.strftime("%Y-%m-%d"),
            "street":      f"{random.randint(100,9999)} {random.choice(STREETS)}",
            "city":        city,
            "state":       state,
            "zip":         f"{random.randint(10000,99999)}",
            "job":         random.choice(JOBS),
            "city_pop":    random.randint(5_000, 3_000_000),
            "lat":         round(random.uniform(25.0, 48.0), 4),
            "long":        round(random.uniform(-122.0, -71.0), 4),
        })
    df = pd.DataFrame(rows)
    log.info(f"customers done: {len(df)} rows")
    return df


def generate_cards(customers: pd.DataFrame) -> pd.DataFrame:
    """
    Generate credit cards linked to customers.
    25 % of customers hold 2 cards; the rest hold 1.
    Fields: card_id, customer_id, cc_num, card_type, credit_limit,
            issue_date, expiry_date
    """
    log.info("Generating cards …")
    rows = []
    counter = 1
    for _, cust in customers.iterrows():
        age        = 2024 - int(str(cust["dob"])[:4])
        base_limit = max(1_000, age * random.randint(80, 220))
        n_cards    = 2 if random.random() < 0.25 else 1
        for _ in range(n_cards):
            cc_num = "".join([str(random.randint(0, 9)) for _ in range(16)])
            rows.append({
                "card_id":      f"CARD{counter:05d}",
                "customer_id":  int(cust["customer_id"]),
                "cc_num":       cc_num,
                "card_type":    random.choices(CARD_TYPES, CARD_WGTS)[0],
                "credit_limit": round(base_limit * random.uniform(0.8, 1.2), -2),
                "issue_date":   (datetime(2015, 1, 1) +
                                 timedelta(days=random.randint(0, 2000))).strftime("%Y-%m-%d"),
                "expiry_date":  (datetime(2025, 1, 1) +
                                 timedelta(days=random.randint(0, 1460))).strftime("%Y-%m-%d"),
            })
            counter += 1
    df = pd.DataFrame(rows)
    log.info(f"cards done: {len(df)} rows")
    return df


def generate_merchants(n: int) -> pd.DataFrame:
    """
    Generate n synthetic merchants.
    Fields: merchant_id, merchant, category, merch_lat, merch_long,
            merch_city, merch_state, risk_rating
    """
    log.info(f"Generating {n} merchants …")
    rows = []
    for i in range(1, n + 1):
        city, state = random.choice(CITIES)
        name = f"fraud_{random.choice(MERCH_PFX).lower()}_{random.choice(MERCH_SFX).lower()}"
        rows.append({
            "merchant_id":  f"MERCH{i:04d}",
            "merchant":     name,
            "category":     random.choice(CATEGORIES),
            "merch_lat":    round(random.uniform(25.0, 48.0), 4),
            "merch_long":   round(random.uniform(-122.0, -71.0), 4),
            "merch_city":   city,
            "merch_state":  state,
            "risk_rating":  random.choices(["low","medium","high"], [0.6, 0.3, 0.1])[0],
        })
    df = pd.DataFrame(rows)
    log.info(f"merchants done: {len(df)} rows")
    return df


def generate_transactions(cards: pd.DataFrame,
                           merchants: pd.DataFrame,
                           n: int,
                           fraud_rate: float,
                           batch_size: int = 500_000,
                           output_path: str = None) -> pd.DataFrame:
    """
    Generate n transactions linked to cards and merchants using vectorized
    numpy batching for memory efficiency at 10M+ row scale.

    Amount distribution: log-normal, fraud skews higher (Fed Reserve 2022).
    Fraud rate ≈ fraud_rate (default 2 %).

    If output_path is provided, writes directly to CSV in batches (avoids
    holding all 10M rows in memory at once). Returns an empty sentinel df.

    Fields: trans_id, trans_num, card_id, merchant_id, category, amt,
            trans_date_trans_time, unix_time, is_fraud
    """
    log.info(f"Generating {n:,} transactions in batches of {batch_size:,} …")
    card_ids   = cards["card_id"].values
    merch_ids  = merchants["merchant_id"].values
    categories = merchants["category"].values      # index-aligned with merch_ids

    start_unix = int(datetime(2019, 1, 1).timestamp())
    end_unix   = int(datetime(2020, 12, 31).timestamp())

    first_batch   = True
    total_fraud   = 0

    for batch_start in range(0, n, batch_size):
        batch_n    = min(batch_size, n - batch_start)
        card_idx   = np.random.randint(0, len(card_ids),  batch_n)
        merch_idx  = np.random.randint(0, len(merch_ids), batch_n)
        is_fraud   = (np.random.random(batch_n) < fraud_rate).astype(np.int8)
        unix_times = np.random.randint(start_unix, end_unix, batch_n)
        trans_ids  = np.arange(batch_start + 1, batch_start + batch_n + 1)

        # Vectorised amount: fraud shifts lognormal mean upward
        amt = np.where(
            is_fraud == 1,
            np.clip(np.random.lognormal(5.0, 1.2, batch_n).round(2), 0.5, 5_000.0),
            np.clip(np.random.lognormal(3.5, 1.0, batch_n).round(2), 0.5, 3_000.0),
        ).astype(np.float32)

        df_batch = pd.DataFrame({
            "trans_id":              trans_ids,
            "trans_num":             [f"{x:016x}" for x in trans_ids],
            "card_id":               card_ids[card_idx],
            "merchant_id":           merch_ids[merch_idx],
            "category":              categories[merch_idx],
            "amt":                   amt,
            "trans_date_trans_time": pd.to_datetime(
                                         unix_times, unit="s"
                                     ).strftime("%Y-%m-%d %H:%M:%S"),
            "unix_time":             unix_times,
            "is_fraud":              is_fraud,
        })

        total_fraud += int(is_fraud.sum())

        if output_path:
            mode   = "w" if first_batch else "a"
            header = first_batch
            df_batch.to_csv(output_path, index=False, mode=mode, header=header)
            first_batch = False
        else:
            if first_batch:
                accumulated = df_batch
                first_batch = False
            else:
                accumulated = pd.concat([accumulated, df_batch], ignore_index=True)

        pct = min(batch_start + batch_n, n) / n * 100
        log.info(f"  transactions batch {batch_start+batch_n:,}/{n:,} ({pct:.0f}%)")
        print(f"  {min(batch_start+batch_n, n):>11,} / {n:,}  ({pct:.0f}%)")

    actual_rate = total_fraud / n * 100
    log.info(f"transactions done: {n:,} rows, fraud_rate={actual_rate:.2f}%")

    if output_path:
        # Return minimal sentinel with necessary columns for validation to pass
        return pd.DataFrame({
            "trans_id": [1],
            "is_fraud": [0],
            "card_id": [cards["card_id"].iloc[0]], # Add a dummy card_id
            "merchant_id": [merchants["merchant_id"].iloc[0]], # Add a dummy merchant_id
            "note": ["written to file"]
        })
    return accumulated


# ── Pipeline checks ────────────────────────────────────────────────────────
def validate_tables(customers, cards, merchants, transactions) -> None:
    """
    Run referential integrity and schema checks.
    Raises AssertionError if any check fails.
    """
    log.info("Running validation checks …")

    # Primary-key uniqueness
    assert customers["customer_id"].nunique() == len(customers), \
        "customer_id not unique"
    assert cards["card_id"].nunique() == len(cards), \
        "card_id not unique"
    assert merchants["merchant_id"].nunique() == len(merchants), \
        "merchant_id not unique"
    assert transactions["trans_id"].nunique() == len(transactions), \
        "trans_id not unique"

    # Referential integrity
    assert cards["customer_id"].isin(customers["customer_id"]).all(), \
        "cards.customer_id has orphaned references"
    assert transactions["card_id"].isin(cards["card_id"]).all(), \
        "transactions.card_id has orphaned references"
    assert transactions["merchant_id"].isin(merchants["merchant_id"]).all(), \
        "transactions.merchant_id has orphaned references"

    # No missing values in key columns
    for col in ["customer_id","first","last","gender","dob"]:
        assert customers[col].notna().all(), f"customers.{col} has nulls"
    assert transactions["is_fraud"].isin([0, 1]).all(), \
        "is_fraud must be binary 0/1"

    log.info("All validation checks passed")
    print("All validation checks passed.")


# ── Main ───────────────────────────────────────────────────────────────────
def main():
    make_output_dir(OUTPUT_DIR)

    customers = generate_customers(N_CUSTOMERS)
    cards     = generate_cards(customers)
    merchants = generate_merchants(N_MERCHANTS)

    # Save reference tables first (small — fit in memory fine)
    paths = {
        "customers":    f"{OUTPUT_DIR}/customers.csv",
        "cards":        f"{OUTPUT_DIR}/cards.csv",
        "merchants":    f"{OUTPUT_DIR}/merchants.csv",
        "transactions": f"{OUTPUT_DIR}/transactions.csv",
    }
    customers.to_csv(paths["customers"], index=False)
    cards.to_csv(paths["cards"],         index=False)
    merchants.to_csv(paths["merchants"], index=False)
    log.info("Reference tables saved.")

    # Transactions: write directly to CSV in batches (10M rows, avoid OOM)
    print(f"\nGenerating {N_TRANSACTIONS:,} transactions in batches …")
    transactions_sentinel = generate_transactions(
        cards, merchants,
        n=N_TRANSACTIONS,
        fraud_rate=FRAUD_RATE,
        batch_size=BATCH_SIZE,
        output_path=paths["transactions"],
    )

    # Validate using a sample (full 10M validation would be slow)
    print("\nRunning validation on reference tables …")
    validate_tables(customers, cards, merchants, transactions_sentinel)

    # Summary
    print("\n=== Data Creation Summary ===")
    for name in ["customers", "cards", "merchants", "transactions"]:
        sz = os.path.getsize(paths[name])
        unit = "GB" if sz > 1e9 else "MB"
        val  = sz / (1e9 if unit == "GB" else 1e6)
        print(f"  {name:<14} {val:>7.2f} {unit}   →  {paths[name]}")

    total_gb = sum(os.path.getsize(p) for p in paths.values()) / 1e9
    print(f"\n  Total CSV: {total_gb:.3f} GB")
    print("\n  ➜ Run convert_to_parquet.py to convert all tables to Parquet format.")
    log.info(f"All CSV files saved. Total: {total_gb:.3f} GB")
    log.info("=== credit_card_creation.py complete ===")
    log.info("Next step: run convert_to_parquet.py")


if __name__ == "__main__":
    main()


Generating 10,000,000 transactions in batches …
      500,000 / 10,000,000  (5%)
    1,000,000 / 10,000,000  (10%)
    1,500,000 / 10,000,000  (15%)
    2,000,000 / 10,000,000  (20%)
    2,500,000 / 10,000,000  (25%)
    3,000,000 / 10,000,000  (30%)
    3,500,000 / 10,000,000  (35%)
    4,000,000 / 10,000,000  (40%)
    4,500,000 / 10,000,000  (45%)
    5,000,000 / 10,000,000  (50%)
    5,500,000 / 10,000,000  (55%)
    6,000,000 / 10,000,000  (60%)
    6,500,000 / 10,000,000  (65%)
    7,000,000 / 10,000,000  (70%)
    7,500,000 / 10,000,000  (75%)
    8,000,000 / 10,000,000  (80%)
    8,500,000 / 10,000,000  (85%)
    9,000,000 / 10,000,000  (90%)
    9,500,000 / 10,000,000  (95%)
   10,000,000 / 10,000,000  (100%)

Running validation on reference tables …
All validation checks passed.

=== Data Creation Summary ===
  customers         0.10 MB   →  ./data/customers.csv
  cards             0.09 MB   →  ./data/cards.csv
  merchants         0.06 MB   →  ./data/merchants.csv
  transact

In [ ]:
import pandas as pd

transactions = pd.read_csv('./data/transactions.csv')
transactions.head()

,trans_id,trans_num,card_id,merchant_id,category,amt,trans_date_trans_time,unix_time,is_fraud
0,1,0000000000000001,CARD01127,MERCH0127,entertainment,1.73,2019-04-12 03:24:57,1555039497,0
1,2,0000000000000002,CARD00861,MERCH0757,shopping_net,52.48,2020-03-15 19:54:24,1584302064,0
2,3,0000000000000003,CARD01131,MERCH0161,food_dining,94.24,2019-04-25 12:41:19,1556196079,0
3,4,0000000000000004,CARD01096,MERCH0309,travel,18.12,2020-06-22 05:32:16,1592803936,0
4,5,0000000000000005,CARD01045,MERCH0625,health_fitness,28.78,2019-02-05 15:16:40,1549379800,0


In [ ]:
customers = pd.read_csv('./data/customers.csv')
customers.head()

,customer_id,first,last,gender,dob,street,city,state,zip,job,city_pop,lat,long
0,1,James,Jones,M,1997-05-08,1779 Hill St,San Diego,CA,81482,Nurse,2481705,34.7042,-120.4803
1,2,Joseph,Miller,M,1982-10-01,9028 Washington Blvd,Denver,CO,38893,Technician,2476559,31.3984,-77.6657
2,3,Brian,Miller,M,1977-06-09,5614 Oak Ave,Phoenix,AZ,22156,Architect,410657,33.2565,-104.4583
3,4,Patricia,Gonzalez,F,1996-08-18,1391 Lake Dr,Houston,TX,48427,Professor,1521807,38.2791,-86.0668
4,5,Kevin,Brown,M,1964-05-03,6327 Pine Rd,San Diego,CA,69429,Scientist,687220,33.5142,-111.3151


In [ ]:
merchants = pd.read_csv('./data/merchants.csv')
merchants.head()

,merchant_id,merchant,category,merch_lat,merch_long,merch_city,merch_state,risk_rating
0,MERCH0001,fraud_national_store,personal_care,43.0346,-75.0059,Nashville,TN,medium
1,MERCH0002,fraud_pioneer_hub,grocery_pos,35.1573,-76.3853,Fort Worth,TX,low
2,MERCH0003,fraud_golden_plus,kids_pets,32.2373,-108.1926,Columbus,OH,low
3,MERCH0004,fraud_star_express,misc_net,43.6541,-99.7255,Chicago,IL,medium
4,MERCH0005,fraud_national_store,kids_pets,35.2948,-109.9563,Columbus,OH,low


In [ ]:
cards= pd.read_csv('./data/cards.csv')
cards.head()

,card_id,customer_id,cc_num,card_type,credit_limit,issue_date,expiry_date
0,CARD00001,1,6945964883192134,Visa,4200.0,2015-09-17,2027-12-06
1,CARD00002,2,9949732033192227,Amex,6900.0,2020-01-09,2025-04-17
2,CARD00003,3,7675481386667548,Visa,4500.0,2018-05-24,2026-03-15
3,CARD00004,4,4268774186520575,Mastercard,3000.0,2018-02-20,2026-01-27
4,CARD00005,5,375830945154809,Visa,11200.0,2018-08-07,2027-09-08


In [ ]:
"""
convert_to_parquet.py
DS 4320 Project 1 – Detecting Credit Card Fraud
Author: Emujin Batzorig

Run this script locally (requires pyarrow) to convert all CSV tables to
standard Parquet format with Snappy compression.

Install dependency:
    pip install pyarrow

Usage:
    python convert_to_parquet.py

Input:  data/*.csv
Output: data/*.parquet  (replaces CSV as primary storage format)
"""

import logging
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

logging.basicConfig(
    filename="convert_to_parquet.log",
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
)
log = logging.getLogger(__name__)
log.info("=== convert_to_parquet.py started ===")

DATA_DIR = "./data"

# Explicit schemas ensure correct types regardless of CSV parsing quirks
SCHEMAS = {
    "customers": pa.schema([
        pa.field("customer_id", pa.int32()),
        pa.field("first",       pa.string()),
        pa.field("last",        pa.string()),
        pa.field("gender",      pa.string()),
        pa.field("dob",         pa.string()),
        pa.field("street",      pa.string()),
        pa.field("city",        pa.string()),
        pa.field("state",       pa.string()),
        pa.field("zip",         pa.string()),
        pa.field("job",         pa.string()),
        pa.field("city_pop",    pa.int32()),
        pa.field("lat",         pa.float32()),
        pa.field("long",        pa.float32()),
    ]),
    "cards": pa.schema([
        pa.field("card_id",      pa.string()),
        pa.field("customer_id",  pa.int32()),
        pa.field("cc_num",       pa.string()),
        pa.field("card_type",    pa.string()),
        pa.field("credit_limit", pa.float32()),
        pa.field("issue_date",   pa.string()),
        pa.field("expiry_date",  pa.string()),
    ]),
    "merchants": pa.schema([
        pa.field("merchant_id",  pa.string()),
        pa.field("merchant",     pa.string()),
        pa.field("category",     pa.string()),
        pa.field("merch_lat",    pa.float32()),
        pa.field("merch_long",   pa.float32()),
        pa.field("merch_city",   pa.string()),
        pa.field("merch_state",  pa.string()),
        pa.field("risk_rating",  pa.string()),
    ]),
    "transactions": pa.schema([
        pa.field("trans_id",              pa.int64()),
        pa.field("trans_num",             pa.string()),
        pa.field("card_id",               pa.string()),
        pa.field("merchant_id",           pa.string()),
        pa.field("category",              pa.string()),
        pa.field("amt",                   pa.float32()),
        pa.field("trans_date_trans_time", pa.string()),
        pa.field("unix_time",             pa.int64()),
        pa.field("is_fraud",              pa.int8()),
    ]),
}


def csv_to_parquet(name: str, chunksize: int = 500_000) -> None:
    """
    Convert a single CSV file to Parquet using chunked reading for memory efficiency.
    Large tables (transactions) are written in chunks to avoid OOM errors.
    """
    csv_path     = os.path.join(DATA_DIR, f"{name}.csv")
    parquet_path = os.path.join(DATA_DIR, f"{name}.parquet")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    schema  = SCHEMAS[name]
    writer  = None
    total   = 0

    try:
        for chunk in pd.read_csv(csv_path, chunksize=chunksize, low_memory=False):
            # Cast to explicit types
            for field in schema:
                col = field.name
                if col not in chunk.columns:
                    raise ValueError(f"Missing column '{col}' in {name}.csv")
                if pa.types.is_integer(field.type):
                    chunk[col] = pd.to_numeric(chunk[col], errors='coerce').fillna(0).astype(int)
                elif pa.types.is_floating(field.type):
                    chunk[col] = pd.to_numeric(chunk[col], errors='coerce').fillna(0.0)
                else:
                    chunk[col] = chunk[col].fillna("").astype(str)

            table = pa.Table.from_pandas(chunk[schema.names], schema=schema,
                                         preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(parquet_path, schema,
                                          compression='snappy')
            writer.write_table(table)
            total += len(chunk)

    finally:
        if writer:
            writer.close()

    csv_size     = os.path.getsize(csv_path)
    parquet_size = os.path.getsize(parquet_path)
    ratio        = csv_size / parquet_size
    log.info(f"{name}: {total:,} rows | CSV {csv_size/1e6:.1f} MB → "
             f"Parquet {parquet_size/1e6:.1f} MB (ratio {ratio:.1f}×)")
    print(f"  ✓ {name:<14} {total:>11,} rows  |  "
          f"CSV {csv_size/1e6:>7.1f} MB  →  Parquet {parquet_size/1e6:>7.1f} MB  "
          f"({ratio:.1f}× compression)")


def main():
    print("Converting CSVs to Parquet (Snappy compression) …\n")
    total_csv     = 0
    total_parquet = 0

    for name in ["customers", "cards", "merchants", "transactions"]:
        csv_to_parquet(name)
        total_csv     += os.path.getsize(os.path.join(DATA_DIR, f"{name}.csv"))
        total_parquet += os.path.getsize(os.path.join(DATA_DIR, f"{name}.parquet"))

    print(f"\nTotal  CSV     : {total_csv/1e9:.3f} GB")
    print(f"Total  Parquet : {total_parquet/1e9:.3f} GB")
    print("\nAll Parquet files written to ./data/")
    log.info(f"Done. Total parquet: {total_parquet/1e9:.3f} GB")
    log.info("=== convert_to_parquet.py complete ===")


if __name__ == "__main__":
    main()

Converting CSVs to Parquet (Snappy compression) …

  ✓ customers            1,000 rows  |  CSV     0.1 MB  →  Parquet     0.1 MB  (1.9× compression)
  ✓ cards                1,264 rows  |  CSV     0.1 MB  →  Parquet     0.1 MB  (1.7× compression)
  ✓ merchants              800 rows  |  CSV     0.1 MB  →  Parquet     0.0 MB  (3.0× compression)
  ✓ transactions    10,000,000 rows  |  CSV   951.0 MB  →  Parquet   321.7 MB  (3.0× compression)

Total  CSV     : 0.951 GB
Total  Parquet : 0.322 GB

All Parquet files written to ./data/
